# Event Impact Analysis

This notebook explores the core research question:
**Do specific news event types reliably move crypto prices?**

Prerequisites:
```bash
python scripts/ingest.py          # Collect articles + prices
python scripts/process.py         # Classify events
```

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from src.storage.database import Database
from src.analysis.event_impact import EventImpactAnalyzer
from src.analysis.signal_generator import SignalGenerator
from src.analysis.backtester import Backtester
from src.analysis.narrative_tracker import NarrativeTracker
from src.config import load_config

config = load_config('../config.yaml')
db = Database(config['database']['path'])

print('Database stats:')
for table, count in db.get_stats().items():
    print(f'  {table}: {count:,}')

## 1. Event Classification Distribution

First, check how articles are distributed across event categories.

In [ ]:
events = db.get_events(limit=10000)
df_events = pd.DataFrame(events)

print(f'Total events: {len(df_events)}')
print()
print('Category distribution:')
cat_counts = df_events['category'].value_counts()
for cat, count in cat_counts.items():
    print(f'  {cat:<20} {count:>5}  ({count/len(df_events):.1%})')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cat_counts.plot.barh(ax=axes[0], color='#6366f1')
axes[0].set_title('Events by Category')
axes[0].set_xlabel('Count')

df_events['severity'].value_counts().sort_index().plot.bar(ax=axes[1], color='#f59e0b')
axes[1].set_title('Events by Severity')
axes[1].set_xlabel('Severity')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 2. Price Impact Analysis

For each event category, measure the average price change at +1h, +4h, +24h.

In [ ]:
analyzer = EventImpactAnalyzer(db, config)
results = analyzer.analyze_by_category(min_severity=1)

if results:
    rows = []
    for r in results:
        rows.append({
            'Category': r.category,
            'Window': f'{r.window_hours}h',
            'Avg Move %': r.avg_move_pct,
            'Median Move %': r.median_move_pct,
            'Win Rate': r.win_rate,
            'N': r.sample_size,
            'p-value': r.p_value,
            'Significant': r.significant,
        })
    df_impact = pd.DataFrame(rows)
    display(df_impact.style.format({
        'Avg Move %': '{:+.2f}%',
        'Median Move %': '{:+.2f}%',
        'Win Rate': '{:.1%}',
        'p-value': '{:.4f}',
    }).applymap(lambda x: 'background-color: #d4edda' if x == True else '', subset=['Significant']))
else:
    print('No impact data. Ensure you have both events and price data.')

## 3. Impact Visualization

Bar chart of average price moves by category for each time window.

In [ ]:
if results:
    windows = sorted(set(r.window_hours for r in results))
    fig, axes = plt.subplots(1, len(windows), figsize=(6 * len(windows), 6), sharey=True)
    if len(windows) == 1:
        axes = [axes]
    
    for ax, window in zip(axes, windows):
        wr = [r for r in results if r.window_hours == window]
        wr.sort(key=lambda r: r.avg_move_pct)
        
        cats = [r.category for r in wr]
        moves = [r.avg_move_pct for r in wr]
        colors = ['#22c55e' if r.significant and r.avg_move_pct > 0
                  else '#ef4444' if r.significant and r.avg_move_pct < 0
                  else '#999999' for r in wr]
        
        ax.barh(cats, moves, color=colors, height=0.6)
        for i, r in enumerate(wr):
            sig = ' ***' if r.significant else ''
            ax.text(r.avg_move_pct + 0.05, i, f'n={r.sample_size}{sig}', va='center', fontsize=9)
        
        ax.axvline(x=0, color='black', linewidth=0.8)
        ax.set_title(f'{window}h Window', fontsize=13, fontweight='bold')
        ax.set_xlabel('Avg Price Move (%)')
        ax.grid(axis='x', alpha=0.3)
    
    plt.suptitle('Event Impact: Average Price Move by Category', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

## 4. Significant Findings

Filter to only statistically significant results.

In [ ]:
if results:
    significant = [r for r in results if r.significant]
    if significant:
        print('STATISTICALLY SIGNIFICANT FINDINGS:')
        print('=' * 60)
        for r in sorted(significant, key=lambda x: abs(x.avg_move_pct), reverse=True):
            direction = '+' if r.avg_move_pct > 0 else ''
            print(f'  {r.category} (sev>={r.min_severity}) → '
                  f'avg {r.window_hours}h move: {direction}{r.avg_move_pct:.1f}%, '
                  f'n={r.sample_size}, p={r.p_value:.4f}')
    else:
        print('No significant findings yet. Need more data (sample size >= 10).')

## 5. Backtesting

Simulate trading based on event signals. How would this have performed historically?

In [ ]:
bt = Backtester(db, config)
result = bt.run(min_severity=2)

if result.total_trades > 0:
    print(bt.generate_report(min_severity=2))
    
    # Equity curve
    pnls = [t.pnl_pct * bt.position_size for t in result.trades]
    cum_pnl = np.cumsum(pnls)
    
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(cum_pnl, color='#6366f1', linewidth=1.5)
    ax.fill_between(range(len(cum_pnl)), cum_pnl, alpha=0.15, color='#6366f1')
    ax.axhline(y=0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Trade #')
    ax.set_ylabel('Cumulative Return (%)')
    ax.set_title('Backtest Equity Curve')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No trades generated. Need events with matching price data.')

## 6. Narrative Analysis

What themes are dominating crypto news? Which narratives have momentum?

In [ ]:
tracker = NarrativeTracker(db, config)
snapshots = tracker.update_narratives(days=90)

if snapshots:
    print(tracker.generate_report(days=90))
else:
    print('No narratives detected yet.')